In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# LAMMPS LOG PARSER
# ============================================================

def extract_thermo_data(
    log_path,
    cutoff_step=None,
    last_ns=None,
    timestep_fs=1.0,
):
    """
    Extract thermo quantities from a LAMMPS log file.

    Parameters
    ----------
    log_path : str
        Path to log.lammps

    cutoff_step : int or None
        Use data after this step only

    last_ns : float or None
        Extract only last X ns

    timestep_fs : float
        MD timestep in fs

    Returns
    -------
    thermo_dict : dict
        Dictionary containing all thermo quantities
    """

    with open(log_path, "r") as f:
        lines = f.readlines()

    # --------------------------------------------------------
    # Find thermo header
    # --------------------------------------------------------

    header = None
    start_idx = None

    for i, line in enumerate(lines):
        if line.strip().startswith("Step"):
            cols = line.split()

            if "PotEng" in cols or "pe" in cols:
                header = cols
                start_idx = i
                break

    if header is None:
        raise RuntimeError(f"Thermo header not found in {log_path}")

    # --------------------------------------------------------
    # Store thermo data
    # --------------------------------------------------------

    thermo = {key: [] for key in header}

    for line in lines[start_idx + 1:]:

        parts = line.split()

        if len(parts) != len(header):
            continue

        try:
            step = int(parts[0])
        except:
            continue

        for key, value in zip(header, parts):

            try:
                thermo[key].append(float(value))
            except:
                pass

    # Convert to numpy arrays
    for key in thermo:
        thermo[key] = np.array(thermo[key])

    # --------------------------------------------------------
    # Filtering
    # --------------------------------------------------------

    steps = thermo["Step"]

    if cutoff_step is not None:

        mask = steps >= cutoff_step

    elif last_ns is not None:

        max_step = steps.max()

        steps_target = int((last_ns * 1e6) / timestep_fs)

        cutoff = max_step - steps_target

        mask = steps >= cutoff

    else:

        mask = np.ones_like(steps, dtype=bool)

    for key in thermo:
        thermo[key] = thermo[key][mask]

    return thermo


# ============================================================
# ENERGY ANALYSIS
# ============================================================

def compute_adsorption_energy(
    close_log,
    peg_log,
    csh_log,
    cutoff_step=None,
    average_last_n=None,
    two_box=False,
):
    """
    Compute adsorption energy:

    E_ads = E_close - (E_peg + E_csh)
    """

    close_data = extract_thermo_data(
        close_log,
        cutoff_step=cutoff_step,
    )

    peg_data = extract_thermo_data(
        peg_log,
        cutoff_step=cutoff_step,
    )

    csh_data = extract_thermo_data(
        csh_log,
        cutoff_step=cutoff_step,
    )

    # --------------------------------------------------------
    # Detect PE column
    # --------------------------------------------------------

    pe_key = "PotEng" if "PotEng" in close_data else "pe"

    close_pe = close_data[pe_key]
    peg_pe = peg_data[pe_key]
    csh_pe = csh_data[pe_key]

    # --------------------------------------------------------
    # Average strategy
    # --------------------------------------------------------

    if average_last_n is not None:

        E_close = np.mean(close_pe[-average_last_n:])
        E_peg = np.mean(peg_pe[-average_last_n:])
        E_csh = np.mean(csh_pe[-average_last_n:])

    else:

        E_close = np.mean(close_pe)
        E_peg = np.mean(peg_pe)
        E_csh = np.mean(csh_pe)

    if two_box:
        E_ads = E_close - E_peg
    else:
        E_ads = E_close - (E_peg + E_csh)

    # --------------------------------------------------------
    # Print summary
    # --------------------------------------------------------

    print("\n==============================")
    print("Adsorption Energy Summary")
    print("==============================")

    print(f"Close System     : {E_close:.6f}")
    print(f"PEG + Water      : {E_peg:.6f}")
    print(f"CSH + Water      : {E_csh:.6f}")
    print(f"Separated System : {E_peg + E_csh:.6f}")

    print("------------------------------")
    print(f"Adsorption Energy: {E_ads:.6f} kcal/mol")
    print("==============================\n")

    return {
        "close": close_data,
        "peg": peg_data,
        "csh": csh_data,
        "E_ads": E_ads,
    }


# ============================================================
# PLOTTING
# ============================================================

def plot_thermo_grid(data_dict):

    quantities = [
        "Temp",
        "PotEng",
        "TotEng",
        "Press",
    ]

    systems = [
        ("close", "Close"),
        ("peg", "PEG + Water"),
        ("csh", "CSH + Water"),
    ]

    fig, axs = plt.subplots(
        len(quantities),
        len(systems),
        figsize=(15, 12),
    )

    for row, quantity in enumerate(quantities):

        for col, (key, title) in enumerate(systems):

            ax = axs[row, col]

            data = data_dict[key]

            if quantity not in data:
                ax.set_visible(False)
                continue

            ax.plot(data[quantity])

            ax.set_title(f"{title} : {quantity}")

            ax.set_xlabel("Frame")
            ax.set_ylabel(quantity)

    plt.tight_layout()
    plt.show()


def plot_thermo_grid_together(data_dict, quantities, systems, fig, axs, name=None):

    for row, quantity in enumerate(quantities):

        for col, (key, title) in enumerate(systems):

            ax = axs[row, col]

            data = data_dict[key]

            if quantity not in data:
                ax.set_visible(False)
                continue

            ax.plot(data[quantity], label=name)

            ax.set_title(f"{title} : {quantity}")

            ax.legend()
            ax.set_xlabel("Frame")
            ax.set_ylabel(quantity)

# ============================================================
# USER INPUT
# ============================================================

BASE = Path(
    ""
    "MDSetup/example/mechanical_properties/"
    "0_CSH_transfer/CSH_surface/try/"
    "2_combined_new"
)



In [ ]:
RUN = "0_run12_pcff"

########### 1_1 ###############
close_log = BASE / RUN / "1_1-12E/S1/log.lammps"
peg_log   = BASE / RUN / "1_1-12E/S2/NPT/log.lammps"
csh_log   = BASE / RUN / "S3/NPT/log.lammps"
results_11n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


########### 2_1 ###############
close_log = BASE / RUN / "2_1-8E/S1/log.lammps"
peg_log   = BASE / RUN / "2_1-8E/S2/NPT/log.lammps"
csh_log   = BASE / RUN / "S3/NPT/log.lammps"
results_21n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


########### 3_1 ###############
close_log = BASE / RUN / "3_1-6E/S1/log.lammps"
peg_log   = BASE / RUN / "3_1-6E/S2/NPT/log.lammps"
csh_log   = BASE / RUN / "S3/NPT/log.lammps"
results_31n = compute_adsorption_energy(close_log=close_log, peg_log=peg_log, csh_log=csh_log, cutoff_step=10, average_last_n=100)


quantities = ["Temp", "PotEng", "TotEng", "Press"]
systems = [("close", "Close"), ("peg", "PEG + Water"), ("csh", "CSH + Water")]
fig, axs = plt.subplots(len(quantities), len(systems), figsize=(15, 12))



plot_thermo_grid_together(results_11n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_21n, quantities, systems, fig, axs)
plot_thermo_grid_together(results_31n, quantities, systems, fig, axs)

plt.tight_layout()
plt.show()


In [ ]:





import numpy as np
import matplotlib.pyplot as plt

# Settings
nsteps = 2000          # number of points from the end
smooth_window = 100    # moving average window
fit_window = 500       # local linear fit window

systems = [("1:1", results_11n), ("2:1", results_21n), ("3:1", results_31n)]
phases = ["close", "peg", "csh"]

# Plot
fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
for i, (ratio, results) in enumerate(systems):
    for j, phase in enumerate(phases):
        ax = axes[i, j]        
        
        # Data
        pe = np.asarray(results[phase]["PotEng"][-nsteps:])
        x = np.arange(len(pe))
        
        # Raw trajectory
        ax.plot(x, pe, color="red", alpha=0.35, lw=1, label="Raw PE")
        
        # Moving average
        pe_smooth = np.convolve(pe, np.ones(smooth_window) / smooth_window, mode="valid")
        x_smooth = np.arange(len(pe_smooth)) + smooth_window // 2
        ax.plot(x_smooth, pe_smooth, color="darkred", lw=2, label="Moving Avg")

        # Piecewise linear fits        
        slopes = []
        for start in range(0, len(pe) - fit_window + 1, fit_window):
            xs = x[start:start + fit_window]
            ys = pe[start:start + fit_window]
            z = np.polyfit(xs, ys, 1)
            p = np.poly1d(z)
            slopes.append(z[0])
            ax.plot(xs, p(xs), 'k--', lw=2)

        # # Summary statistics        
        # avg_slope = np.mean(slopes)
        # final_slope = slopes[-1]
        # ax.set_title(f"{phase.upper()} ({ratio})\n" f"Final slope = {final_slope:.4f}", fontsize=10)

        slope_text = ", ".join([f"{s:.3f}" for s in slopes])
        ax.set_title(f"{phase.upper()} ({ratio})\n{slope_text}", fontsize=12)

        if i == 2: ax.set_xlabel("Time (steps)")
        if j == 0: ax.set_ylabel("PE (kcal/mol)")
        ax.grid(alpha=0.3)

# # Column headers
# for j, phase in enumerate(phases):
#     axes[0, j].set_title(f"{phase.upper()}", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
ll = []
for i in range(-1000, 3001, 10):
    ll.append(results_31n['close']['Lz'][-3500:][results_31n['close']['Press'][-3500:]<=i].mean())

plt.plot(range(-1000, 3001, 10), ll)
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()

In [ ]:
ll = []
for i in range(-1000, 3001, 10):
    ll.append(results_31n['csh']['Lz'][-3500:][results_31n['csh']['Press'][-3500:]<=i].mean())

plt.plot(range(-1000, 3001, 10), ll)
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()

In [ ]:
ll = []
for i in range(-1000, 3001, 10):
    ll.append(results_31n['peg']['Lz'][-3500:][results_31n['peg']['Press'][-3500:]<=i].mean())

plt.plot(range(-1000, 3001, 10), ll)
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()

In [ ]:
ll = []
for i in range(-1000, 1001, 10):
    ll.append(results_11n['peg']['Lz'][-3500:][(results_11n['peg']['Press'][-3500:]<=i) * (results_11n['peg']['Press'][-3500:]>=(i-100))].mean())

plt.plot(range(-1000, 1001, 10), ll, label='Mean (P±100)')
plt.axhline(results_11n['peg']['Lz'][-3500:].mean(), color='red', linestyle='--', label='Overall Mean')

plt.axhline(results_11n['peg']['Lz'][-3500:][(results_11n['peg']['Press'][-3500:]<=250) * (results_11n['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--', label='Mean (0±250)')
plt.axhline(results_11n['peg']['Lz'][-3500:][(results_11n['peg']['Press'][-3500:]<=100) * (results_11n['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--', label='Mean (0±100)')
plt.legend()
plt.axhline(101.19874377275974, color='blue', linestyle='--')
plt.title("Lz vs Pressure (PEG) - 1:1-12E")
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.grid()
plt.show()

In [ ]:
ll = []
for i in range(-1000, 1001, 10):
    ll.append(results_21n['peg']['Lz'][-3500:][(results_21n['peg']['Press'][-3500:]<=i) * (results_21n['peg']['Press'][-3500:]>=(i-100))].mean())

plt.plot(range(-1000, 1001, 10), ll, label='Mean (P±100)')
plt.axhline(results_21n['peg']['Lz'][-3500:].mean(), color='red', linestyle='--', label='Overall Mean')

plt.axhline(results_21n['peg']['Lz'][-3500:][(results_21n['peg']['Press'][-3500:]<=250) * (results_21n['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--', label='Mean (0±250)')
plt.axhline(results_21n['peg']['Lz'][-3500:][(results_21n['peg']['Press'][-3500:]<=100) * (results_21n['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--', label='Mean (0±100)')
plt.axhline(101.32629980915769, color='blue', linestyle='--')
plt.legend()
plt.title("Lz vs Pressure (PEG) - 2:1-8E")
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.grid()
plt.show()

In [ ]:
ll = []
for i in range(-1000, 1001, 10):
    ll.append(results_31n['peg']['Lz'][-3500:][(results_31n['peg']['Press'][-3500:]<=i) * (results_31n['peg']['Press'][-3500:]>=(i-100))].mean())

plt.plot(range(-1000, 1001, 10), ll, label='Mean (P±100)')
plt.axhline(results_31n['peg']['Lz'][-3500:].mean(), color='red', linestyle='--', label='Overall Mean')
plt.axhline(results_31n['peg']['Lz'][-3500:][(results_31n['peg']['Press'][-3500:]<=250) * (results_31n['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--', label='Mean (0±250)')
plt.axhline(results_31n['peg']['Lz'][-3500:][(results_31n['peg']['Press'][-3500:]<=100) * (results_31n['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--', label='Mean (0±100)')
plt.axhline(101.3997940027139, color='blue', linestyle='--')


plt.legend()
plt.title("Lz vs Pressure (PEG) - 3:1-6E")
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.grid()
plt.show()



In [ ]:
ll = []
for i in range(-4000, 101, 10):
    ll.append(results_31n['csh']['Lz'][-3500:][(results_31n['csh']['Press'][-3500:]<=i) * (results_31n['csh']['Press'][-3500:]>=(i-100))].mean())

plt.plot(range(-4000, 101, 10), ll, label='Mean (P±100)')

plt.axhline(results_31n['csh']['Lz'][-3500:].mean(), color='red', linestyle='--', label='Overall Mean')
plt.axhline(results_31n['csh']['Lz'][-3500:][(results_31n['csh']['Press'][-3500:]<=250) * (results_31n['csh']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--', label='Mean (0±250)')
plt.axhline(results_31n['csh']['Lz'][-3500:][(results_31n['csh']['Press'][-3500:]<=100) * (results_31n['csh']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--', label='Mean (0±100)')
plt.axhline(58.189971708333346, color='blue', linestyle='--', label='previously used Lz')

plt.legend()

plt.title("Lz vs Pressure (S3-CSH)")

plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.grid()
plt.show()

In [ ]:
# plt.plot([101.19,101.21],[101.19,101.21])
plt.axhline(results_11n['peg']['Lz'][-3500:].mean(), color='red', linestyle='--')
plt.axhline(results_11n['peg']['Lz'][-3500:][(results_11n['peg']['Press'][-3500:]<=250) * (results_11n['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--')
plt.axhline(results_11n['peg']['Lz'][-3500:][(results_11n['peg']['Press'][-3500:]<=100) * (results_11n['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--')
plt.axhline(101.19978360666667, color='blue', linestyle='--')
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()


# plt.plot([101.3,101.35],[101.3,101.35])
plt.axhline(results_21n['peg']['Lz'][-3500:].mean(), color='red', linestyle='--')
plt.axhline(results_21n['peg']['Lz'][-3500:][(results_21n['peg']['Press'][-3500:]<=250) * (results_21n['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--')
plt.axhline(results_21n['peg']['Lz'][-3500:][(results_21n['peg']['Press'][-3500:]<=100) * (results_21n['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--')
plt.axhline(101.31636158999999, color='blue', linestyle='--')
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()



# plt.plot([101.38,101.42],[101.38,101.42])
plt.axhline(results_31n['peg']['Lz'][-3500:].mean(), color='red', linestyle='--')
plt.axhline(results_31n['peg']['Lz'][-3500:][(results_31n['peg']['Press'][-3500:]<=250) * (results_31n['peg']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--')
plt.axhline(results_31n['peg']['Lz'][-3500:][(results_31n['peg']['Press'][-3500:]<=100) * (results_31n['peg']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--')
plt.axhline(101.40595109666667, color='blue', linestyle='--')

plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()


# plt.plot([58.1,58.2],[58.1,58.2])
plt.axhline(results_31n['csh']['Lz'][-3500:].mean(), color='red', linestyle='--')
plt.axhline(results_31n['csh']['Lz'][-3500:][(results_31n['csh']['Press'][-3500:]<=250) * (results_31n['csh']['Press'][-3500:]>=-250)].mean(), color='green', linestyle='--')
plt.axhline(results_31n['csh']['Lz'][-3500:][(results_31n['csh']['Press'][-3500:]<=100) * (results_31n['csh']['Press'][-3500:]>=-100)].mean(), color='k', linestyle='--')
plt.axhline(58.12622, color='blue', linestyle='--')
plt.xlabel("Pressure (bar)")
plt.ylabel("Lz (Angstrom)")
plt.title("Lz vs Pressure")
plt.grid()
plt.show()




In [ ]:

print('250')
print((results_11n['peg']['Lz'][-3500:][(results_11n['peg']['Press'][-3500:]<=250) * (results_11n['peg']['Press'][-3500:]>=-250)].mean()))
print((results_21n['peg']['Lz'][-3500:][(results_21n['peg']['Press'][-3500:]<=250) * (results_21n['peg']['Press'][-3500:]>=-250)].mean()))
print((results_31n['peg']['Lz'][-3500:][(results_31n['peg']['Press'][-3500:]<=250) * (results_31n['peg']['Press'][-3500:]>=-250)].mean()))
print((results_31n['csh']['Lz'][-3500:][(results_31n['csh']['Press'][-3500:]<=250) * (results_31n['csh']['Press'][-3500:]>=-250)].mean()))

print('100')
print((results_11n['peg']['Lz'][-3500:][(results_11n['peg']['Press'][-3500:]<=100) * (results_11n['peg']['Press'][-3500:]>=-100)].mean()))
print((results_21n['peg']['Lz'][-3500:][(results_21n['peg']['Press'][-3500:]<=100) * (results_21n['peg']['Press'][-3500:]>=-100)].mean()))
print((results_31n['peg']['Lz'][-3500:][(results_31n['peg']['Press'][-3500:]<=100) * (results_31n['peg']['Press'][-3500:]>=-100)].mean()))
print((results_31n['csh']['Lz'][-3500:][(results_31n['csh']['Press'][-3500:]<=100) * (results_31n['csh']['Press'][-3500:]>=-100)].mean()))

print('avg')
print(((results_11n['peg']['Lz'][-3500:][(results_11n['peg']['Press'][-3500:]<=250) * (results_11n['peg']['Press'][-3500:]>=-250)].mean()) + (results_11n['peg']['Lz'][-3500:][(results_11n['peg']['Press'][-3500:]<=100) * (results_11n['peg']['Press'][-3500:]>=-100)].mean()))/2)
print(((results_21n['peg']['Lz'][-3500:][(results_21n['peg']['Press'][-3500:]<=250) * (results_21n['peg']['Press'][-3500:]>=-250)].mean()) + (results_21n['peg']['Lz'][-3500:][(results_21n['peg']['Press'][-3500:]<=100) * (results_21n['peg']['Press'][-3500:]>=-100)].mean()))/2)
print(((results_31n['peg']['Lz'][-3500:][(results_31n['peg']['Press'][-3500:]<=250) * (results_31n['peg']['Press'][-3500:]>=-250)].mean()) + (results_31n['peg']['Lz'][-3500:][(results_31n['peg']['Press'][-3500:]<=100) * (results_31n['peg']['Press'][-3500:]>=-100)].mean()))/2)
print(((results_31n['csh']['Lz'][-3500:][(results_31n['csh']['Press'][-3500:]<=250) * (results_31n['csh']['Press'][-3500:]>=-250)].mean()) + (results_31n['csh']['Lz'][-3500:][(results_31n['csh']['Press'][-3500:]<=100) * (results_31n['csh']['Press'][-3500:]>=-100)].mean()))/2)
